In [25]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
import string

In [27]:
with open("shakespeare.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("Dataset length:", len(text))
print(text[:500])  # preview

Dataset length: 5359444
*** START OF THE PROJECT GUTENBERG EBOOK 100 ***




The Complete Works of William Shakespeare

by William Shakespeare




                    Contents

    THE SONNETS
    ALL’S WELL THAT ENDS WELL
    THE TRAGEDY OF ANTONY AND CLEOPATRA
    AS YOU LIKE IT
    THE COMEDY OF ERRORS
    THE TRAGEDY OF CORIOLANUS
    CYMBELINE
    THE TRAGEDY OF HAMLET, PRINCE OF DENMARK
    THE FIRST PART OF KING HENRY THE FOURTH
    THE SECOND PART OF KING HENRY THE FOURTH
    THE LIFE OF KING HENRY THE FIFTH
  


In [41]:
# convert text to lowercase
text = text.lower()

# remove punctuation
text = text.translate(str.maketrans('', '', string.punctuation))

# Spilt text into lines"
lines = text.split('\n')

# remove empty lines
lines = [line.strip() for line in lines if line.strip() != ""]

# reduce dataset size for making training faster
lines = lines[:20000]

print("Total lines used:", len(lines))

Total lines used: 20000


In [43]:
tokenizer = Tokenizer()

tokenizer.fit_on_texts(lines)

total_words = len(tokenizer.word_index) + 1
print("Total vocabulary size:", total_words)

Total vocabulary size: 10720


In [45]:
input_sequences = []

for line in lines:
    
    # Convert line to token sequence
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    # Generate sequences
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print("Total sequences:", len(input_sequences))

Total sequences: 105242


In [46]:
max_sequence_len = max([len(x) for x in input_sequences])

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
)

# Split into X and y
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Input shape:", X.shape)
print("Output shape:", y.shape)

Input shape: (105242, 18)
Output shape: (105242,)


In [51]:
model = Sequential()

# Embedding layer
model.add(Embedding(total_words, 100))

# LSTM layer
model.add(LSTM(150))

# Output layer
model.add(Dense(total_words, activation='softmax'))

# Compile model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [53]:
early_stop = EarlyStopping(
    monitor='loss',
    patience=3,
    restore_best_weights=True
)
history = model.fit(
    X,
    y,
    epochs=8,
    batch_size=64,
    callbacks=[early_stop]
)

Epoch 1/8
1645/1645 ━━━━━━━━━━━━━━━━━━━━ 208s 119ms/step - accuracy: 0.0318 - loss: 7.1072
Epoch 2/8
1645/1645 ━━━━━━━━━━━━━━━━━━━━ 123s 75ms/step - accuracy: 0.0590 - loss: 6.3911
Epoch 3/8
1645/1645 ━━━━━━━━━━━━━━━━━━━━ 117s 71ms/step - accuracy: 0.0878 - loss: 6.0198
Epoch 4/8
1645/1645 ━━━━━━━━━━━━━━━━━━━━ 122s 74ms/step - accuracy: 0.0999 - loss: 5.7535
Epoch 5/8
1645/1645 ━━━━━━━━━━━━━━━━━━━━ 121s 74ms/step - accuracy: 0.1134 - loss: 5.5065
Epoch 6/8
1645/1645 ━━━━━━━━━━━━━━━━━━━━ 124s 75ms/step - accuracy: 0.1220 - loss: 5.2864
Epoch 7/8
1645/1645 ━━━━━━━━━━━━━━━━━━━━ 131s 79ms/step - accuracy: 0.1311 - loss: 5.0706
Epoch 8/8
1645/1645 ━━━━━━━━━━━━━━━━━━━━ 123s 75ms/step - accuracy: 0.1417 - loss: 4.8448


In [55]:
def generate_text(seed_text, next_words=20):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len-1,
            padding='pre'
        )
        predicted = np.argmax(model.predict(token_list), axis=-1)
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

In [57]:
print(generate_text("to be or not", 15))
print(generate_text("love is", 15))
print(generate_text("the king", 15))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 826ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
to be or not to be a fool to be a maid and to be a maid in the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━

In [62]:
print(generate_text("And being frank", 15))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
And being frank in the city of the forest of arden and the people and the gods be


## Dataset Link
https://www.gutenberg.org/files/100/100-0.txt